# TD-clip retrain — sweep runs over seeds × del-lims (wandb)

Retrains pretrained checkpoints under a grid of **seeds × TD clips** (`limit_delta`,
here called *del-lim*). Each (seed, del-lim) is a separate wandb run named
`Retrain_Seed{seed}_{tag}` (mirroring training's `Run_Seed{seed}`), retrained with
the **same hyperparameters as `run_training.ipynb`**, and saved as a full checkpoint
bundle under `results/retrain/<del_lim>/checkpoints_retrain_<seed>/`.

This notebook only **runs & saves** — plotting/averaging/inference live in the
companion plotting notebook.

In [1]:
# ── Imports ───────────────────────────────────────────────────────────────
import os, json, warnings
import numpy as np
import torch

from agents.ppo import PPOAgent

try:
    import wandb
    WANDB_AVAILABLE = True
except ImportError:
    WANDB_AVAILABLE = False
    warnings.warn('wandb not installed — logging disabled.')

print('Imports OK  |  wandb available:', WANDB_AVAILABLE)

Imports OK  |  wandb available: True


/Users/charithapalika/Desktop/Lab projects/Appetite modelling/FoodRL/FoodRL_cleaned_copies/utils/utils.py:3: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


## Config

In [2]:
# ── Sweep grid: seeds × del-lims (EDIT THESE) ─────────────────────────────
SEEDS    = [2026]#[0, 7, 64, 27, 42, 100, 107, 1997, 2003, 2026]                       # seeds that have results/checkpoints_<seed>/
# DEL_LIMS = [None, -5e-3, -3e-3, -1e-3, -5e-4, -3e-4,-1e-4, -5e-5, -3e-5, -1e-5, -5e-6, -3e-6,
#             -1e-6, 1e-6, 3e-6, 5e-6, 1e-5, 3e-5, 5e-5, 1e-4, 3e-4, 5e-4, 1e-3, 3e-3, 5e-3 ] #[None, -1e-5, 1e-5]       # limit_delta values. None = no clip.
                                     #   +c upper clip (caps +TD), -c lower clip (floors -TD)
DEL_LIMS = [5e-2]#[5e-2, 1e-1]
CKPT_NAME        = 'last'            # source checkpoint per seed: 'last' | 'best' | 'epN'
RETRAIN_EPISODES = 500

# ── Retrain hyperparameters — MATCHED to run_training.ipynb ────────────────
TRAIN_KW = dict(
    rollout_steps      = 256,
    ppo_epochs         = 4,
    minibatch_size     = 64,
    shared_ac_lr       = 1e-4,
    # lr_schedule        = {'type': 'exponential', 'final_frac': 0.1},
    log_every_episodes = 10,
)

# ── wandb ─────────────────────────────────────────────────────────────────
LOG_WANDB    = True
PROJECT_NAME = 'Food RL'             # same project as training
RETRAIN_ROOT = 'results/retrain_500ep'
os.makedirs(RETRAIN_ROOT, exist_ok=True)

if LOG_WANDB and not WANDB_AVAILABLE:
    LOG_WANDB = False
    print('wandb not available — LOG_WANDB forced False.')

# ── del-lim -> readable tag / folder name ─────────────────────────────────
def dl_tag(dl):
    if dl is None or dl == 0:
        return 'noclip'
    return f'{"pos" if dl > 0 else "neg"}_{abs(float(dl)):g}'

print(f'Seeds    : {SEEDS}')
print(f'Del-lims : {DEL_LIMS}  ->  tags {[dl_tag(d) for d in DEL_LIMS]}')
print(f'Grid     : {len(SEEDS) * len(DEL_LIMS)} runs x {RETRAIN_EPISODES} eps')
print(f'Save root: {RETRAIN_ROOT}/<del_lim>/checkpoints_retrain_<seed>/')

Seeds    : [2026]
Del-lims : [0.05]  ->  tags ['pos_0.05']
Grid     : 1 runs x 500 eps
Save root: results/retrain_500ep/<del_lim>/checkpoints_retrain_<seed>/


## Run the sweep

In [3]:
# ── Retrain every (del_lim, seed) ─────────────────────────────────────────
import time

def retrain_one(seed, dl):
    src_meta = f'results/training_checkpoints/checkpoints_{seed}/ppo_agent_{CKPT_NAME}_meta.json'
    if not os.path.exists(src_meta):
        print(f'  SKIP seed {seed}: no checkpoint at {src_meta}')
        return None

    agent = PPOAgent.from_checkpoint(src_meta, device='cpu')
    agent.args['limit_delta'] = dl                     # apply the del-lim (clip)

    run = None
    if LOG_WANDB:
        run = wandb.init(
            project=PROJECT_NAME,
            name=f'Retrain_500ep_Seed{seed}_{dl_tag(dl)}',
            reinit=True,
            config={'seed': seed, 'limit_delta': dl, 'del_lim_tag': dl_tag(dl),
                    'source_ckpt': src_meta, 'retrain_episodes': RETRAIN_EPISODES,
                    **TRAIN_KW},
        )

    ckpt_dir = os.path.join(RETRAIN_ROOT, dl_tag(dl), 'checkpoints_retrain')
    
    tlog = agent.train(num_episodes=RETRAIN_EPISODES, log_wandb=LOG_WANDB,
                           printing=False, checkpoint_every=100, checkpoint_dir=ckpt_dir,
                           **TRAIN_KW)

    # ── save bundle: results/retrain/<del_lim>/checkpoints_retrain_<seed>/ ──
    # save_final appends '_<seed>' to the dir, giving the exact folder we want.
    ckpt_dir = os.path.join(RETRAIN_ROOT, dl_tag(dl), 'checkpoints_retrain')
    prefix   = agent.save_final(ckpt_dir, log_wandb=LOG_WANDB)      # .../checkpoints_retrain_<seed>/ppo_agent_last_*
    out_dir  = os.path.dirname(prefix)
    agent.save_training_log(os.path.join(out_dir, 'training_log.npz'))
    with open(os.path.join(out_dir, 'retrain_info.json'), 'w') as f:
        json.dump({'seed': seed, 'limit_delta': dl, 'del_lim_tag': dl_tag(dl),
                   'source_ckpt': src_meta, 'retrain_episodes': RETRAIN_EPISODES,
                   'train_kw': TRAIN_KW}, f, indent=2)
    if run is not None:
        wandb.finish()
    return {'seed': seed, 'dl': dl, 'out_dir': out_dir,
            'last_return': float(tlog['reward'][-1]),
            'roll50_return': float(np.mean(tlog['reward'][-50:])),
            'roll50_food': float(np.mean(tlog['consumption'][-50:]))}

summary = []
t_all = time.time()
for dl in DEL_LIMS:
    for seed in SEEDS:
        print(f'=== del_lim={dl} ({dl_tag(dl)})  |  seed {seed} ===')
        t0 = time.time()
        res = retrain_one(seed, dl)
        if res is None:
            continue
        summary.append(res)
        print(f'  [{time.time()-t0:5.1f}s] roll50 return={res["roll50_return"]:8.2f}  '
              f'food={res["roll50_food"]:5.2f}  -> {res["out_dir"]}')
print(f'\nSweep done in {time.time()-t_all:.1f}s — {len(summary)} runs saved under {RETRAIN_ROOT}/')

=== del_lim=0.05 (pos_0.05)  |  seed 2026 ===
[FoodEnv] Loading  'glucose'  from  'food_dataset/serum_glucose.csv'
           min=70.0758  max=142.5123   foods=19   time_points=500
[FoodEnv] Loading  'peptides'  from  'food_dataset/small_peptides_absorbed.csv'
           min=0.0000  max=0.0018   foods=19   time_points=500
[FoodEnv] Loading  'fatty_acids'  from  'food_dataset/fatty_acids_absorbed.csv'
           min=0.0000  max=0.0007   foods=19   time_points=500
[FoodEnv] Ready — 19 foods | 3 time-series nutrients | 0 cumulative nutrients
[FoodEnv] Loading shadow nutrient 'fullness'  from  'food_dataset/fullness.csv'
[FoodEnv] Loading shadow nutrient 'hunger'  from  'food_dataset/hunger.csv'
[FoodEnv] Loading shadow nutrient 'cck'  from  'food_dataset/cck.csv'
[FoodEnv] Loading shadow nutrient 'ghrelin'  from  'food_dataset/ghrelin.csv'
[FoodEnv] Loading shadow nutrient 'glp_1'  from  'food_dataset/glp_1.csv'
[FoodEnv] Loading shadow nutrient 'pyy'  from  'food_dataset/pyy.csv'
[FoodEn

wandb: Currently logged in as: charithapalika (charitha_palika) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


wandb: Detected [agents] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


train/actor_loss,▁▅▄▅▇▆▆▄▅▆▅▅█▆▆▅▄▆▆▄▇▆▇▇▆▄▇▆▆▅▅▄▆▅▅▆▆▅▆▇
train/consumption,▂▅▄█▄▃▄▄▃▅▄▄▄▃▄▄▃▁▃▃▄▄▃▄▄▄▄▄▃▄▅▅▅▄▆▄▄▄▄▄
train/critic_loss,▄▄▁▄▂▁▂▂▁▃▂▃▄▁▁▂▃▁▂▂▃▂▂▁▂▂▂▂▂█▂▂▂▁▂▃▁▃▂▃
train/distance,▂▁▃█▂▂▁▄▄▃▄▅▄▃▄▄▄▃▃▃▃▄▅▃▁▂▃▁▄▄▄▆▅▃▃▂▄▃▃▄
train/entropy,█▄▅▅▄▃▃▄▃▃▂▃▂▃▁▂▁▂▂▃▂▃▃▃▃▅▂▃▃▃▄▃▄▄▄▃▃▃▄▃
train/reward,▆▅▁█▅█▄▅▆▆▄▆▃▅▄▄▆▆▄▄▆▄▅▇▅▅█▅▄▄▂▄▆▅▇▅▆▆▅▄
train/reward_rolling_avg,▇█▇▆▆▅▅▅▅▅▅▃▃▂▂▄▃▃▃▃▄▄▄▄▅▅▅▅▅▄▂▂▂▂▁▂▃▄▄▅
train/shared_ac_lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/actor_loss,-0.00038
train/consumption,13
train/critic_loss,0.20834


  [ 79.3s] roll50 return=  163.18  food=12.06  -> results/retrain_500ep/pos_0.05/checkpoints_retrain_2026

Sweep done in 79.3s — 1 runs saved under results/retrain_500ep/


In [4]:
# ── Recap table of what was saved ─────────────────────────────────────────
import pandas as pd
if summary:
    df = pd.DataFrame(summary)
    df['del_lim_tag'] = [dl_tag(d) for d in df['dl']]
    display(df[['del_lim_tag', 'seed', 'roll50_return', 'roll50_food', 'out_dir']].round(3))
else:
    print('No runs saved — check SEEDS have results/checkpoints_<seed>/ bundles.')

,del_lim_tag,seed,roll50_return,roll50_food,out_dir
0,pos_0.05,2026,163.18,12.06,results/retrain_500ep/pos_0.05/checkpoints_ret...
